In [ ]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
# from model.systemModel                  import SystemModel as systemModel
from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json
import package
import sys
import math


package.include()
outputDir = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")
# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

##### HI COMPONENT AND SYSTEM

In [ ]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

package.include()
outputDir       = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")
cr1, cr2, cv2   = preData.loadH5File(dir=outputDir + "data/", name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file

# -------------------------------
# Handle to make the length of 
# each component consistently
cr1 = preData.nanFilling(dfFiltered=cr1, validColumn=preData.crusher1Value)
cr2 = preData.nanFilling(dfFiltered=cr2, validColumn=preData.crusher2Value)
cv2 = preData.nanFilling(dfFiltered=cv2, validColumn=preData.conveyorValue + preData.crusher2ConveyorValue)
# insertedCr1, insertedCr2,  insertedCv2   = self.fillForTheSameLength(data         = [withHourCr1,withHourCr2,withHourCv2], 
#                                                                      validColumn  = [self.crusher1Value, self.crusher2Value, self.conveyorValue + self.crusher2ConveyorValue])

if preData.config["splitTrainTest"]:
    target_time = pd.Timestamp('2023-10-01 00:00:00')

    rowCr1 = cr1.index[pd.to_datetime(cr1['tiempo']) == target_time][0]

    rowCr2 = cr2.index[pd.to_datetime(cr2['tiempo']) == target_time][0]

    rowCv2 = cv2.index[pd.to_datetime(cv2['tiempo']) == target_time][0]

    # Display the row
    print(rowCr1, rowCr2, rowCv2)

    trainCr1 = cr1.iloc[:rowCr1]
    trainCr2 = cr2.iloc[:rowCr2]
    trainCv2 = cv2.iloc[:rowCv2]
else:
    trainCr1 = cr1
    trainCr2 = cr2
    trainCv2 = cv2

# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(trainCr1.copy())
withHourCr2             = preData.addHours(trainCr2.copy())
withHourCv2             = preData.addHours(trainCv2.copy())

# if self.config["component"]  == "system":

# else:
#     insertedCr1 = withHourCr1
#     insertedCr2 = withHourCr2
#     insertedCv2 = withHourCv2
# --------------------------------
# Sumarry data (option)
# _,mofCr1                = self.sumarryDataFromFile(withHourCr1, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")
# _                       = self.sumarryDataFromFile(withHourCr2, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")
# _                       = self.sumarryDataFromFile(withHourCv2, printUsefulMOF = True, printMOFTable = True,typeOfTime= "hours")

withIDResults, mofOrder, shutdownMask = preData.attachIDPerEvent(df = [withHourCr1,withHourCr2,withHourCv2])
# withIDCr1, withIDCr2, withIDCv2, IdSys

print("--------------------WITHOUT MAINTENANCE--------------------------")
print(withIDResults["cr1"]['trajIDWithoutM'])
print(withIDResults["cr2"]['trajIDWithoutM'])
print(withIDResults["cv2"]['trajIDWithoutM'])

print("------------------WITH MAINTENANCE----------------------------")
print(withIDResults["cr1"]['trajIDWithM'])
print(withIDResults["cr2"]['trajIDWithM'])
print(withIDResults["cv2"]['trajIDWithM'])
# print(withIDResults["sys"])

addedData               = preData.addition_var(withIDResults["cr1"]) 
handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1

# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData,            headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDResults["cr2"], headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDResults["cv2"], headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1, shutdownMask["cr1"])                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2, shutdownMask["cr2"])                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2, shutdownMask["cv2"])                        # Add the time to the data

# aiLogger.info("Export the file to test!!!")
# withTimeDataCr1.to_excel("IDCrusher1.xlsx", index=False)

# -----------------------------
# Add the cycle and RUL of system
systemData                      = pd.DataFrame()
systemData['hours']             = withTimeDataCr1["hours"]
systemData['trajIDWithoutM']    = withIDResults["sysIDWithoutM"]
systemData['trajIDWithM']       = withIDResults["sysIDWithM"]
withTimeDataSys                 = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
timeLabel               = ["tFailureWithMaintenance", "tFailureWithoutMaintenance"]
dataCr1                 = withTimeDataCr1[timeLabel + handledValues]
cycleCr1                = withTimeDataCr1["tFailureWithoutMaintenance"]
rulCr1                  = withTimeDataCr1["rul"]
trajectIDCr1            = withTimeDataCr1["trajIDWithM"]
timeCr1                 = withTimeDataCr1["hours"]

dataCr2                 = withTimeDataCr2[timeLabel + preData.crusher2Value]
cycleCr2                = withTimeDataCr2["tFailureWithoutMaintenance"]
rulCr2                  = withTimeDataCr2["rul"]
trajectIDCr2            = withTimeDataCr2["trajIDWithM"]

dataCv2                 = withTimeDataCv2[timeLabel + preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["tFailureWithoutMaintenance"]
rulCv2                  = withTimeDataCv2["rul"]
trajectIDCv2            = withTimeDataCv2["trajIDWithM"]

systemRUL               = withTimeDataSys["rul"]
timeLabel               = ["hours", "tFailureWithMaintenance", "tFailureWithoutMaintenance", "tBreak"]

# print(timeCr1)
# print(timeCr2)
# print(timeCv2)
# print(insertedCr1Sys["hours"])



### Crusher1

In [3]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

In [ ]:
myConfig["component"] = "Cr1"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr1"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel37/")
model.load_weights(model_dir = outputDir + "checkpoint/bestModel/")

In [ ]:
results             = {}
time                = {}
results["cr1"]      = []
results["rulCr1"]   = []
time["timeCr1"]     = []
listRUL             = []

probsAll        = []
binaryMaskAll   = []
trajectID       = trajectIDCr1.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1, seqTimeCr1,seqRUL           = preData.getTensorData(dataList   =    [dataCr1, timeCr1,rulCr1], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1)
    # print(predictedRul)
    results["cr1"].append(pd.Series(tf.squeeze(zCr1).numpy()))
    results["rulCr1"].append(predictedRul)
    time["timeCr1"].append(seqTimeCr1[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCr1"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCr1"])
print("Crusher 1 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))

# temp_df = pd.concat([rulCr1, rulCr2, rulCv2], axis=1)
# # print(np.max(np.concatenate(rulCr1, rulCr2 , rulCv2)), np.min(np.concatenate(rulCr1 , rulCr2, rulCv2)))
# print(temp_df.max().max(),temp_df.min().min())

In [ ]:
print(time["timeCr1"][0].shape)

### Crusher2

### ORIGINAL RESULTS

##### CR1 SENSOR SELECTION

In [ ]:
timeLabel               = [ "tFailureWithMaintenance", "tFailureWithoutMaintenance", ]
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(timeLabel + handledValues):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)

##### CR2 SENSOR SELECTION

#### HI AND RUL VISUALIZATION

In [ ]:
fig = {}
for idxKey,resultKeys in enumerate(results.keys()):
    fig[resultKeys]     = go.Figure()
    current_time        = 0
    # print(math.floor(idxKey/2) + 1)
    for idx, signal in enumerate(results[resultKeys]):
        # print(signal)
        signal          = tf.squeeze(signal).numpy()
        # print(idx)
        # print(signal)
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[resultKeys].add_trace(go.Scatter(
                                                x    = time_vector,
                                                y    = signal,
                                                mode = 'lines',
                                                name = f'{idx}. ' 
        ))
    # Customize layout for readability
    fig[resultKeys].update_layout(
                                    title       = resultKeys,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[resultKeys].show()


### FILTERED RESULTS